# OpenAI Realtime + LangSmith

This notebook keeps the workshop-facing pieces visible: the Realtime session shape, the LangSmith tracing setup, and the final voice loop. Local mic/speaker transport, audio recording, event handling, and cleanup live in `voice_demo.workshop`.

## 1. Building the Agent Brains

The speech-to-speech brain is the Realtime session configuration: instructions, audio settings, and the weather tool schema the model can call mid-conversation.

In [ ]:
import os
import uuid

from dotenv import load_dotenv

from voice_demo.workshop import run_openai_realtime_agent

load_dotenv()

PROJECT = "voice-workshop-s2s"
MODEL = os.getenv("REALTIME_MODEL", "gpt-realtime-2")
SAMPLE_RATE = 24_000

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."

In [ ]:
SYSTEM_PROMPT = """You are a friendly voice assistant who can look up the
weather for any city. Keep replies short, conversational, and free of
formatting. When the user asks about weather, call lookup_weather once per
city, then summarize the result in one or two spoken sentences."""

WEATHER_TOOL = {
    "type": "function",
    "name": "lookup_weather",
    "description": "Get the current weather for a single city. Call once per city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name, e.g. Paris."}
        },
        "required": ["city"],
    },
}

SESSION = {
    "type": "realtime",
    "instructions": SYSTEM_PROMPT,
    "output_modalities": ["audio"],
    "audio": {
        "input": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "transcription": {"model": "gpt-4o-mini-transcribe"},
            "noise_reduction": {"type": "near_field"},
            "turn_detection": {
                "type": "server_vad",
                "create_response": False,
                "interrupt_response": True,
            },
        },
        "output": {
            "format": {"type": "audio/pcm", "rate": SAMPLE_RATE},
            "voice": "alloy",
        },
    },
    "tools": [WEATHER_TOOL],
    "tool_choice": "auto",
}

## 2. Setting Up Tracing

The OpenAI Realtime runner wraps the connection with LangSmith. The only thing we create here is the thread id that groups this voice session in LangSmith.

In [ ]:
thread_id = str(uuid.uuid4())
thread_id

## 3. Running the Voice Agent

The helper opens the Realtime connection, wraps it for LangSmith, streams microphone audio in, plays model audio out, records both channels, and runs tools when the model asks for weather.

In [ ]:
await run_openai_realtime_agent(
    session=SESSION,
    project=PROJECT,
    model=MODEL,
    thread_id=thread_id,
    sample_rate=SAMPLE_RATE,
)